# Normalização de Banco de Dados (1FN a 6FN)
Este notebook demonstra passo a passo a normalização de um conjunto de dados fictício de pedidos, clientes e produtos.

Vamos evoluir desde uma tabela não normalizada até a 6ª Forma Normal (6FN), mostrando exemplos práticos em Python com pandas.

In [1]:
import pandas as pd
import numpy as np

def show(title, df):
    print(f"\n=== {title} ===")
    display(df)

## Tabela inicial (não normalizada)
A tabela possui múltiplos produtos em colunas (`produto1`, `produto2`), bem como telefones e e-mails em formato de lista separada por ponto e vírgula.

In [2]:

pedidos_raw = pd.DataFrame([
    {
        "pedido_id": 1,
        "data_pedido": "2025-09-20",
        "cliente_nome": "Ana Silva",
        "cliente_endereco": "Rua A, 123",
        "cidade": "São Paulo",
        "estado": "SP",
        "telefones": "11-99999-0001;11-99999-0002",
        "emails": "ana@example.com;ana.silva@empresa.com",
        "produto1": "Lápis",
        "preco1": 1.50,
        "produto2": "Caderno",
        "preco2": 10.00
    },
    {
        "pedido_id": 2,
        "data_pedido": "2025-09-21",
        "cliente_nome": "João Souza",
        "cliente_endereco": "Av. B, 456",
        "cidade": "Campinas",
        "estado": "SP",
        "telefones": "19-98888-1111",
        "emails": "joao@example.com",
        "produto1": "Caneta",
        "preco1": 2.00,
        "produto2": "Caderno",
        "preco2": 10.00
    },
    {
        "pedido_id": 3,
        "data_pedido": "2025-09-22",
        "cliente_nome": "Maria Costa",
        "cliente_endereco": "Rua C, 789",
        "cidade": "São Paulo",
        "estado": "SP",
        "telefones": "11-97777-3333;11-96666-4444",
        "emails": "maria@example.com",
        "produto1": "Borracha",
        "preco1": 0.75,
        "produto2": None,
        "preco2": None
    }
])
show("Tabela inicial (não normalizada)", pedidos_raw)


=== Tabela inicial (não normalizada) ===


,pedido_id,data_pedido,cliente_nome,cliente_endereco,cidade,estado,telefones,emails,produto1,preco1,produto2,preco2
0,1,2025-09-20,Ana Silva,"Rua A, 123",São Paulo,SP,11-99999-0001;11-99999-0002,ana@example.com;ana.silva@empresa.com,Lápis,1.50,Caderno,10.0
1,2,2025-09-21,João Souza,"Av. B, 456",Campinas,SP,19-98888-1111,joao@example.com,Caneta,2.00,Caderno,10.0
2,3,2025-09-22,Maria Costa,"Rua C, 789",São Paulo,SP,11-97777-3333;11-96666-4444,maria@example.com,Borracha,0.75,NaN,NaN


## Primeira Forma Normal (1FN)
- Remove grupos repetidos (produto1, produto2)
- Cria uma linha por produto do pedido

In [3]:
rows = []
product_pairs = [(1, 'produto1', 'preco1'), (2, 'produto2', 'preco2')]
for _, r in pedidos_raw.iterrows():
    for _, produto_col, preco_col in product_pairs:
        produto = r[produto_col]
        preco = r[preco_col]
        if pd.notna(produto):
            rows.append({
                "pedido_id": r["pedido_id"],
                "data_pedido": r["data_pedido"],
                "cliente_nome": r["cliente_nome"],
                "cliente_endereco": r["cliente_endereco"],
                "cidade": r["cidade"],
                "estado": r["estado"],
                "telefones": r["telefones"],
                "emails": r["emails"],
                "produto": produto,
                "preco": preco
            })
pedidos_1fn = pd.DataFrame(rows)
show("Pedidos (1FN)", pedidos_1fn)


=== Pedidos (1FN) ===


,pedido_id,data_pedido,cliente_nome,cliente_endereco,cidade,estado,telefones,emails,produto,preco
0,1,2025-09-20,Ana Silva,"Rua A, 123",São Paulo,SP,11-99999-0001;11-99999-0002,ana@example.com;ana.silva@empresa.com,Lápis,1.50
1,1,2025-09-20,Ana Silva,"Rua A, 123",São Paulo,SP,11-99999-0001;11-99999-0002,ana@example.com;ana.silva@empresa.com,Caderno,10.00
2,2,2025-09-21,João Souza,"Av. B, 456",Campinas,SP,19-98888-1111,joao@example.com,Caneta,2.00
3,2,2025-09-21,João Souza,"Av. B, 456",Campinas,SP,19-98888-1111,joao@example.com,Caderno,10.00
4,3,2025-09-22,Maria Costa,"Rua C, 789",São Paulo,SP,11-97777-3333;11-96666-4444,maria@example.com,Borracha,0.75


## Segunda Forma Normal (2FN)
- Elimina dependência parcial da chave composta (pedido_id, produto).
- Criamos tabelas: `Clientes`, `Pedidos`, `Itens_Pedido`. 

In [4]:
clientes = pedidos_1fn[["cliente_nome","cliente_endereco","cidade","estado","telefones","emails"]].drop_duplicates().reset_index(drop=True)
clientes = clientes.reset_index().rename(columns={"index":"cliente_id"})
clientes["cliente_id"] = clientes["cliente_id"] + 1
clientes = clientes[["cliente_id","cliente_nome","cliente_endereco","cidade","estado","telefones","emails"]]
show("Clientes (2FN)", clientes)

pedidos = pedidos_1fn[["pedido_id","data_pedido","cliente_nome","cliente_endereco"]].drop_duplicates().reset_index(drop=True)
pedidos = pedidos.merge(clientes[["cliente_id","cliente_nome","cliente_endereco"]], on=["cliente_nome","cliente_endereco"], how="left")
pedidos = pedidos[["pedido_id","data_pedido","cliente_id"]]
show("Pedidos (2FN)", pedidos)

itens_pedido = pedidos_1fn[["pedido_id","produto","preco"]].reset_index(drop=True)
show("Itens_Pedido (2FN)", itens_pedido)


=== Clientes (2FN) ===


,cliente_id,cliente_nome,cliente_endereco,cidade,estado,telefones,emails
0,1,Ana Silva,"Rua A, 123",São Paulo,SP,11-99999-0001;11-99999-0002,ana@example.com;ana.silva@empresa.com
1,2,João Souza,"Av. B, 456",Campinas,SP,19-98888-1111,joao@example.com
2,3,Maria Costa,"Rua C, 789",São Paulo,SP,11-97777-3333;11-96666-4444,maria@example.com



=== Pedidos (2FN) ===


,pedido_id,data_pedido,cliente_id
0,1,2025-09-20,1
1,2,2025-09-21,2
2,3,2025-09-22,3



=== Itens_Pedido (2FN) ===


,pedido_id,produto,preco
0,1,Lápis,1.50
1,1,Caderno,10.00
2,2,Caneta,2.00
3,2,Caderno,10.00
4,3,Borracha,0.75


## Terceira Forma Normal (3FN)
- Remove dependências transitivas (`cidade → estado`).
- Criamos tabela `Cidades` e referenciamos por `cidade_id`. 

In [5]:
cidades = clientes[["cidade","estado"]].drop_duplicates().reset_index(drop=True)
cidades = cidades.reset_index().rename(columns={"index":"cidade_id"})
cidades["cidade_id"] = cidades["cidade_id"] + 1
cidades = cidades[["cidade_id","cidade","estado"]]
show("Cidades (3FN)", cidades)

clientes_3fn = clientes.merge(cidades, on=["cidade","estado"], how="left")
clientes_3fn = clientes_3fn[["cliente_id","cliente_nome","cliente_endereco","cidade_id","telefones","emails"]]
show("Clientes (após 3FN)", clientes_3fn)


=== Cidades (3FN) ===


,cidade_id,cidade,estado
0,1,São Paulo,SP
1,2,Campinas,SP



=== Clientes (após 3FN) ===


,cliente_id,cliente_nome,cliente_endereco,cidade_id,telefones,emails
0,1,Ana Silva,"Rua A, 123",1,11-99999-0001;11-99999-0002,ana@example.com;ana.silva@empresa.com
1,2,João Souza,"Av. B, 456",2,19-98888-1111,joao@example.com
2,3,Maria Costa,"Rua C, 789",1,11-97777-3333;11-96666-4444,maria@example.com


## Forma Normal de Boyce-Codd (BCNF)
- Elimina dependências funcionais anômalas.
- Exemplo: `produto → preco`. Criamos tabela `Produtos`. 

In [6]:
produtos = itens_pedido[["produto","preco"]].drop_duplicates().reset_index(drop=True)
produtos = produtos.reset_index().rename(columns={"index":"produto_id"})
produtos["produto_id"] = produtos["produto_id"] + 1
produtos = produtos[["produto_id","produto","preco"]].rename(columns={"preco":"preco_unitario"})
show("Produtos (BCNF)", produtos)

itens_pedido_bcnf = itens_pedido.merge(produtos[["produto_id","produto"]], on="produto", how="left")
itens_pedido_bcnf = itens_pedido_bcnf[["pedido_id","produto_id"]]
show("Itens_Pedido (após BCNF)", itens_pedido_bcnf)


=== Produtos (BCNF) ===


,produto_id,produto,preco_unitario
0,1,Lápis,1.50
1,2,Caderno,10.00
2,3,Caneta,2.00
3,4,Borracha,0.75



=== Itens_Pedido (após BCNF) ===


,pedido_id,produto_id
0,1,1
1,1,2
2,2,3
3,2,2
4,3,4


## Quarta Forma Normal (4FN)
- Elimina dependências multivaloradas.
- Telefones e Emails de clientes são separados em tabelas próprias. 

In [7]:
telefones_rows = []
for _, r in clientes_3fn.iterrows():
    cliente_id = r["cliente_id"]
    telefones_raw = r["telefones"]
    if pd.notna(telefones_raw):
        for t in str(telefones_raw).split(";"):
            t = t.strip()
            if t:
                telefones_rows.append({"cliente_id": cliente_id, "telefone": t})
telefones = pd.DataFrame(telefones_rows)
show("Telefones (4FN)", telefones)

emails_rows = []
for _, r in clientes_3fn.iterrows():
    cliente_id = r["cliente_id"]
    emails_raw = r["emails"]
    if pd.notna(emails_raw):
        for e in str(emails_raw).split(";"):
            e = e.strip()
            if e:
                emails_rows.append({"cliente_id": cliente_id, "email": e})
emails = pd.DataFrame(emails_rows)
show("Emails (4FN)", emails)

clientes_4fn = clientes_3fn[["cliente_id","cliente_nome","cliente_endereco","cidade_id"]]
show("Clientes (após 4FN)", clientes_4fn)


=== Telefones (4FN) ===


,cliente_id,telefone
0,1,11-99999-0001
1,1,11-99999-0002
2,2,19-98888-1111
3,3,11-97777-3333
4,3,11-96666-4444



=== Emails (4FN) ===


,cliente_id,email
0,1,ana@example.com
1,1,ana.silva@empresa.com
2,2,joao@example.com
3,3,maria@example.com



=== Clientes (após 4FN) ===


,cliente_id,cliente_nome,cliente_endereco,cidade_id
0,1,Ana Silva,"Rua A, 123",1
1,2,João Souza,"Av. B, 456",2
2,3,Maria Costa,"Rua C, 789",1


## Quinta Forma Normal (5FN)
- Elimina dependências de junção.
- Exemplo com relação ternária (Fornecedor, Produto, Região). 

In [8]:
fornece = pd.DataFrame([
    {"fornecedor":"F1","produto":"P1","regiao":"Norte"},
    {"fornecedor":"F1","produto":"P2","regiao":"Norte"},
    {"fornecedor":"F2","produto":"P1","regiao":"Sul"},
    {"fornecedor":"F2","produto":"P2","regiao":"Sul"},
])
show("R - Fornece (tripla)", fornece)

fornece_fp = fornece[["fornecedor","produto"]].drop_duplicates().reset_index(drop=True)
fornece_fr = fornece[["fornecedor","regiao"]].drop_duplicates().reset_index(drop=True)
fornece_pr = fornece[["produto","regiao"]].drop_duplicates().reset_index(drop=True)
show("Fornece_FP", fornece_fp)
show("Fornece_FR", fornece_fr)
show("Fornece_PR", fornece_pr)

# Reconstrução por junção
temp = fornece_fp.merge(fornece_pr, on="produto")
recon = temp.merge(fornece_fr, on="fornecedor", how="left", suffixes=('_pr','_fr'))
reg_cols = [c for c in recon.columns if 'regiao' in c]
if reg_cols:
    recon['regiao'] = recon[reg_cols[0]]
recon = recon[['fornecedor','produto','regiao']].drop_duplicates().reset_index(drop=True)
show("Reconstrução de R", recon)


=== R - Fornece (tripla) ===


,fornecedor,produto,regiao
0,F1,P1,Norte
1,F1,P2,Norte
2,F2,P1,Sul
3,F2,P2,Sul



=== Fornece_FP ===


,fornecedor,produto
0,F1,P1
1,F1,P2
2,F2,P1
3,F2,P2



=== Fornece_FR ===


,fornecedor,regiao
0,F1,Norte
1,F2,Sul



=== Fornece_PR ===


,produto,regiao
0,P1,Norte
1,P2,Norte
2,P1,Sul
3,P2,Sul



=== Reconstrução de R ===


,fornecedor,produto,regiao
0,F1,P1,Norte
1,F1,P1,Sul
2,F1,P2,Norte
3,F1,P2,Sul
4,F2,P1,Norte
5,F2,P1,Sul
6,F2,P2,Norte
7,F2,P2,Sul


## Sexta Forma Normal (6FN)
- Decomposição extrema, geralmente usada em cenários temporais.
- Exemplo: histórico de preços de produtos em períodos distintos.

In [9]:
prod_precos = pd.DataFrame([
    {"produto_id": 1, "start":"2025-01-01", "end":"2025-06-30", "preco": 1.40},
    {"produto_id": 1, "start":"2025-07-01", "end": None,       "preco": 1.50},
    {"produto_id": 2, "start":"2025-01-01", "end": None,       "preco": 10.00},
])
show("Produto_Precos (6FN - temporal)", prod_precos)


=== Produto_Precos (6FN - temporal) ===


,produto_id,start,end,preco
0,1,2025-01-01,2025-06-30,1.4
1,1,2025-07-01,NaN,1.5
2,2,2025-01-01,NaN,10.0


## Resumo
- **1FN** → remove grupos repetidos
- **2FN** → remove dependência parcial
- **3FN** → remove dependência transitiva
- **BCNF** → remove dependências funcionais anômalas
- **4FN** → remove dependências multivaloradas
- **5FN** → remove dependências de junção
- **6FN** → granularidade máxima (ex.: temporal)